# Работа с ресурсами

# Менеджер контекста для смены директории (cd)

Напишите класс менеджера контекста ChangeDir, который временно меняет текущую рабочую директорию на заданную. После выхода из контекста рабочая директория должна вернуться к предыдущей.

**Условия:**
1.	При входе в блок with менеджер контекста должен изменить текущую директорию на указанную.
2.	При выходе из блока with менеджер контекста должен вернуть рабочую директорию на исходное значение.
3.	Обработайте ситуацию, когда указанный путь не существует, с выводом сообщения об ошибке.

**Пример:**

```python
import os

print("Начальная директория:", os.getcwd())

with ChangeDir("/path/to/new/directory"):
    print("Внутри менеджера:", os.getcwd())

print("После выхода:", os.getcwd())
```

In [ ]:
import os

class ChangeDir:
    def __init__(self, path):
        self.new_path = path
        self.old_path = None

    def __enter__(self):
        self.old_path = os.getcwd()

        if not os.path.exists(self.new_path):
            raise FileNotFoundError(f"Путь {self.new_path} не существует")

        os.chdir(self.new_path)
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        os.chdir(self.old_path)

        if exc_type is not None:
            print(f"Произошла ошибка: {exc_value}")

        return False

print("Начальная директория:", os.getcwd())

try:
    with ChangeDir("/path/to/new/directory"):
        print("Внутри менеджера:", os.getcwd())
except FileNotFoundError as e:
    print(e)

print("После выхода:", os.getcwd())

Начальная директория: /content
Путь /path/to/new/directory не существует
После выхода: /content


# Перенаправления вывода в файл

Напишите класс менеджера контекста RedirectOutput, который временно перенаправляет стандартный поток вывода stdout в указанный файл. После выхода из контекста вывод должен возвращаться в стандартный поток.

**Условия:**

1.	При входе в блок with менеджер контекста должен перенаправить вывод print в файл, указанный при создании объекта.
2.	При выходе из блока with вывод должен возвращаться в стандартный поток.
3.	Если файл уже существует, вывод должен дописываться к нему, а не перезаписывать его.

**Пример:**
```python
print("Это стандартный вывод")  # Должно выводиться в консоль

with RedirectOutput("output.txt"):
    print("Это вывод в файл")   # Должно записываться в файл "output.txt"

print("Снова стандартный вывод")  # Должно выводиться в консоль
```


In [ ]:
#self.file = open(self.filename, 'a')
#sys.stdout = self.file  # Перенаправляем stdout в файл

In [ ]:
import sys

class RedirectOutput:
    def __init__(self, filename):
        self.filename = filename
        self.original_stdout = sys.stdout

    def __enter__(self):
        self.file = open(self.filename, 'a')
        sys.stdout = self.file
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        sys.stdout = self.original_stdout
        self.file.close()

print("Это стандартный вывод")

with RedirectOutput("output.txt"):
    print("Это вывод в файл")

print("Снова стандартный вывод")

Это стандартный вывод
Снова стандартный вывод


# Замер времени выполнения кода

Напишите класс менеджера контекста Timer, который замеряет время выполнения кода внутри блока with. Менеджер должен выводить время выполнения в консоль по завершении блока. Для замера времени используйте модуль time.

**Условия:**
1. При входе в блок with менеджер контекста должен начинать отсчёт времени.
2. При выходе из блока with менеджер должен выводить в консоль время выполнения кода внутри блока в формате "Время выполнения: X.XXX секунд".
3. Опционально: добавить возможность передавать имя таймера при инициализации, чтобы можно было различать результаты замеров, если их несколько.

**Пример:**
```python
import time

with Timer("Задача 1"):
    time.sleep(1)  # Симуляция работы кода
[Задача 1] Время выполнения: 1.001 секунд
    
with Timer("Задача 2"):
    for i in range(1000000):
        pass
[Задача 2] Время выполнения: 0.034 секунд
```

In [ ]:
import time

class Timer:
    def __init__(self, name="Задача"):
        self.name = name

    def __enter__(self):
        self.start_time = time.time()
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        end_time = time.time()
        elapsed_time = end_time - self.start_time
        print(f"[{self.name}] Время выполнения: {elapsed_time:.3f} секунд")

with Timer("Задача 1"):
    time.sleep(1)

with Timer("Задача 2"):
    for i in range(1000000):
        pass

[Задача 1] Время выполнения: 1.001 секунд
[Задача 2] Время выполнения: 0.042 секунд


# Поглощение исключения

Напишите класс менеджера контекста SuppressExceptions, который подавляет указанные исключения внутри блока with, не прерывая выполнение программы. Если в блоке возникает исключение, которое не входит в список подавляемых, оно должно быть выброшено обычным образом.

**Условия:**
1.	При инициализации менеджера контекста нужно передавать типы исключений, которые будут подавляться.
2.	Если в блоке with возникает исключение из списка подавляемых, оно должно игнорироваться.
3.	Если возникает исключение, не входящее в список, оно должно быть выброшено.
4.	Опционально: после подавления исключения вывести сообщение о том, какое исключение было подавлено.


**Пример:**
```python
with SuppressExceptions(ZeroDivisionError, ValueError):
    print(1 / 0)  # Это исключение будет подавлено

with SuppressExceptions(TypeError):
    print(1 + "2")  # Это исключение будет подавлено

with SuppressExceptions(IndexError):
    print([1, 2, 3][5])  # Это исключение будет подавлено

print("Программа продолжает работать после блока with")
```

In [ ]:
class SuppressExceptions:
    def __init__(self, *exceptions):
        self.exceptions = exceptions

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        if exc_type:
            if exc_type in self.exceptions:
                print(f"Исключение {exc_type.__name__} было подавлено.")
                return True
            else:
                return False

with SuppressExceptions(ZeroDivisionError, ValueError):
    print(1 / 0)

with SuppressExceptions(TypeError):
    print(1 + "2")

with SuppressExceptions(IndexError):
    print([1, 2, 3][5])

print("Программа продолжает работать после блока with")


Исключение ZeroDivisionError было подавлено.
Исключение TypeError было подавлено.
Исключение IndexError было подавлено.
Программа продолжает работать после блока with


# Создание временного файла
Напишите класс менеджера контекста TemporaryFile, который создаёт временный файл при входе в контекст и автоматически удаляет его при выходе. Менеджер должен позволять записывать и читать данные из файла в течение его существования в контексте.

**Условия:**
1.	При входе в блок with менеджер должен создавать временный файл и возвращать его объект для записи и чтения.
2.	При выходе из блока with временный файл должен автоматически удаляться.
3.	Имя файла должно быть уникальным и генерироваться автоматически.

**Пример**
```python
with TemporaryFile() as temp_file:
    temp_file.write(b"Временные данные\n")  # Записываем данные
    temp_file.seek(0)  # Возвращаемся в начало файла
    print(temp_file.read())  # Читаем данные из временного файла

print("Файл автоматически удалён")
```

In [ ]:
import tempfile
import os

class TemporaryFile:
    def __enter__(self):
        self.temp_file = tempfile.NamedTemporaryFile(delete=False)
        return self.temp_file

    def __exit__(self, exc_type, exc_value, traceback):
        self.temp_file.close()
        os.remove(self.temp_file.name)

with TemporaryFile() as temp_file:
    temp_file.write(b"Временные данные\n")
    temp_file.seek(0)
    print(temp_file.read())

print("Файл автоматически удалён")

SyntaxError: bytes can only contain ASCII literal characters (<ipython-input-5-9e73a605cc1b>, line 17)